# Dialforge Cloud Benchmark

One-click benchmark for the Dialforge local AI stack on a **Google Colab T4 GPU**. It tests **Qwen 3 1.7B, 4B and 8B**, **faster-whisper small.en**, **Chatterbox Nano**, plus the complete **STT → LLM → TTS pipeline for all three Qwen models**.

1. Choose **Runtime → Change runtime type → T4 GPU** and save.
2. Click **Runtime → Run all**.
3. Leave the tab open until the report appears below.

The notebook deliberately catches setup/runtime failures and prints a clear diagnostic instead of turning every later box red. No SIP credentials are used and no real calls are made.

In [ ]:
# ONE-CLICK DIALFORGE BENCHMARK
# You normally do not need to edit anything in this cell.
QWEN_MODELS = ['qwen3:1.7b', 'qwen3:4b', 'qwen3:8b']
QWEN_REPEATS = 3
COMPONENT_REPEATS = 3
PIPELINE_TURNS = 5

import os, sys, time, pathlib, shutil, subprocess, urllib.request
from IPython.display import HTML, display

REPORT_DIR = pathlib.Path('/content/dialforge-benchmark')
REPORT_HTML = REPORT_DIR / 'dialforge-benchmark-report.html'
RUNNER = '/content/dialforge_colab_benchmark.py'
OLLAMA_LOG = pathlib.Path('/content/ollama.log')

def run(cmd, *, shell=False, check=True, quiet=False):
    shown = cmd if isinstance(cmd, str) else ' '.join(str(x) for x in cmd)
    print('\n>', shown, flush=True)
    if quiet:
        result = subprocess.run(cmd, shell=shell, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if result.stdout:
            print(result.stdout[-6000:], flush=True)
    else:
        result = subprocess.run(cmd, shell=shell)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {shown}')
    return result

def ollama_ready():
    try:
        import requests
        return requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok
    except Exception:
        return False

def install_ollama():
    existing = shutil.which('ollama')
    if existing:
        print('Ollama already installed:', existing)
        run(['ollama', '--version'])
        return
    # Ollama's current Linux installer extracts zstd-compressed archives.
    if not shutil.which('zstd'):
        raise RuntimeError('zstd is missing even after package setup; cannot install Ollama safely.')
    print('Installing Ollama with the official Linux installer...')
    installer = 'curl --retry 5 --retry-delay 2 --retry-connrefused -fsSL https://ollama.com/install.sh | sh'
    first = run(installer, shell=True, check=False, quiet=True)
    if not shutil.which('ollama'):
        print('First Ollama install attempt did not expose the CLI. Retrying once...')
        time.sleep(3)
        second = run(installer, shell=True, check=False, quiet=True)
        if not shutil.which('ollama'):
            raise RuntimeError(f'Ollama install failed after two official-installer attempts (exit codes {first.returncode}, {second.returncode}).')
    run(['ollama', '--version'])

def pip_install(*packages):
    run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages])

def setup_python_stack():
    pip_install('--upgrade', 'pip', 'wheel', 'setuptools')
    # Match the Dialforge beta Chatterbox runtime and use CUDA 12.4 wheels on Colab T4.
    pip_install('torch==2.6.0', 'torchaudio==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
    pip_install('psutil==7.0.0', 'requests==2.32.5', 'faster-whisper==1.2.0', 'chatterbox-tts')
    pip_install('--force-reinstall', '--no-deps', 'git+https://github.com/resemble-ai/Perth.git@ff1c8ac55a976971245cdd53c18d6131ca00d993')
    pip_install('--force-reinstall', '--no-deps', 'git+https://github.com/resemble-ai/chatterbox.git@5de7a54aa4e5e2baadb0182dde554908b48b85c2')

def start_ollama():
    if ollama_ready():
        print('Ollama server already ready.')
        return
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '127.0.0.1:11434'
    env['OLLAMA_ORIGINS'] = '*'
    log = open(OLLAMA_LOG, 'w')
    process = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(75):
        if ollama_ready():
            print('Ollama server ready.')
            return
        if process.poll() is not None:
            break
        time.sleep(1)
    tail = OLLAMA_LOG.read_text(errors='replace')[-8000:] if OLLAMA_LOG.exists() else '(no Ollama log)'
    print('\n----- Ollama log -----\n' + tail)
    raise RuntimeError('Ollama server could not start.')

BENCHMARK_OK = False
try:
    print('=== Dialforge Cloud Benchmark ===')
    if not shutil.which('nvidia-smi'):
        print('\nGPU NOT DETECTED. Choose Runtime > Change runtime type > T4 GPU, save, then run this cell again.')
    else:
        print('\nGPU detected:')
        run(['nvidia-smi'])
        run(['apt-get', 'update', '-qq'])
        # zstd is required by the current official Ollama Linux installer.
        run(['apt-get', 'install', '-y', '-qq', 'pciutils', 'curl', 'ca-certificates', 'git', 'zstd'])
        install_ollama()
        setup_python_stack()
        # Validate the exact GPU/runtime pieces before downloading the large model files.
        validation = [sys.executable, '-c',
            "import inspect,torch,torchaudio,perth; from chatterbox.tts_turbo import ChatterboxTurboTTS; from faster_whisper import WhisperModel; s=inspect.signature(ChatterboxTurboTTS.from_pretrained); assert 'nano' in s.parameters; assert torch.cuda.is_available(); print('CUDA',torch.version.cuda,'GPU',torch.cuda.get_device_name(0)); print('Chatterbox Nano OK | faster-whisper OK | Perth OK')"]
        run(validation)
        start_ollama()
        urllib.request.urlretrieve('https://raw.githubusercontent.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime/main/benchmarks/dialforge_colab_benchmark.py', RUNNER)
        print('Benchmark runner downloaded:', RUNNER)
        cmd = [sys.executable, RUNNER,
               '--models', *QWEN_MODELS,
               '--pipeline-models', *QWEN_MODELS,
               '--qwen-repeats', str(QWEN_REPEATS),
               '--component-repeats', str(COMPONENT_REPEATS),
               '--pipeline-turns', str(PIPELINE_TURNS),
               '--output-dir', str(REPORT_DIR)]
        result = run(cmd, check=False)
        if REPORT_HTML.exists():
            BENCHMARK_OK = True
            print('\n=== BENCHMARK COMPLETE ===')
            display(HTML(REPORT_HTML.read_text(encoding='utf-8')))
        else:
            print(f'\nBenchmark process ended with code {result.returncode}, but no report was produced.')
except Exception as exc:
    print('\n=== BENCHMARK STOPPED SAFELY ===')
    print(type(exc).__name__ + ':', exc)
    print('Nothing was installed on your PC. This error happened only inside the temporary Colab VM.')

if BENCHMARK_OK:
    print('\nResults include standalone tests and full STT -> LLM -> TTS tests for:')
    for model in QWEN_MODELS:
        print(' -', model)


### Result files
When the run completes, Colab also creates:
- `/content/dialforge-benchmark/dialforge-benchmark-report.html`
- `/content/dialforge-benchmark/dialforge-benchmark-report.json`

These are small result files only. The downloaded AI models remain on Google's temporary machine.